In [4]:
# @title Deploy Agent to Agent Engine
! pip install --upgrade --quiet "google-cloud-aiplatform[agent_engines,adk]"

In [ ]:
# @title Restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)



In [2]:
import vertexai
from vertexai.preview.reasoning_engines import AdkApp

from google.adk.tools import google_search
from google.adk.agents import Agent

import vertexai
from vertexai import types

In [4]:
# @title Set up Project and test the Agent Locally

PROJECT_ID = ! gcloud config get-value project
PROJECT_ID = PROJECT_ID[0]
LOCATION = "us-central1" # @param {type:"string"}
MODEL_LOCATION = "global" # @param {type:"string"}

STAGING_BUCKET = "gs://qwiklabs-gcp-02-082f78356513-agent-staging-08cb277c" # @param {type:"string"}

# define project information manually if the above code didn't workqwiklabs-gcp-02-082f78356513$0
if PROJECT_ID == "(unset)":
  PROJECT_ID = "qwiklabs-gcp-02-082f78356513" # @param {type:"string"}

  # Agent Runtime is regional. Do not init with "global" — AdkApp pickles this
# location and uses it for sessions. Gemini 3.6 is pinned to MODEL_LOCATION below.
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)
print(PROJECT_ID)


search_agent = Agent(
    name="Google",
    model="gemini-3.6-flash",
    description=(
        "You can search Google and return answers from the Internet."
    ),
    instruction=(
        """You are a helpful Chatbot.
        If you don't know something, then Search Google for the answer"""
    ),
    tools=[google_search]
)


app = AdkApp(
    agent=search_agent,
    env_vars={"GOOGLE_CLOUD_LOCATION": MODEL_LOCATION},
)
for event in app.stream_query(
    user_id="USER_ID",
    message="Give me news highlights around the world.",
):
    print(event)

qwiklabs-gcp-02-082f78356513


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:1138: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


{'model_version': 'gemini-3.6-flash', 'content': {'parts': [{'text': "Here are top news highlights around the world:\n\n### **Middle East**\n* **Israeli National Election:** Israelis are heading to the polls in a high-stakes national election, with voter turnout and key foreign policy debates—including relations with Iran—at the center of the outcome.\n* **Jordan Security & Regional Tension:** Security forces in Jordan arrested suspects following a deadly church attack, while Jordan launched retaliatory strikes against targets in Syria. \n\n### **Europe & Diplomacy**\n* **Diplomatic Talks in Moscow:** Russian leadership is hosting regional discussions, including meetings with Qatar's Emir and Iran's President alongside ongoing discussions surrounding localized ceasefires along Ukraine's front lines.\n* **U.K. National Security Charges:** U.K. authorities charged two men with national security offenses following an incident near an air base.\n\n### **The Americas**\n* **Brazil Political

In [5]:
# @title Deploy to Agent Runtime
client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": ["google-cloud-aiplatform[agent_engines,adk]"],
        "staging_bucket": STAGING_BUCKET,
        "env_vars": {"GOOGLE_CLOUD_LOCATION": MODEL_LOCATION},
        "identity_type": types.IdentityType.AGENT_IDENTITY,
        "display_name": "Google News Agent",
        "description": "This agent can search Google for the news.",
    },
)

for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="Give me news highlights around the world.",
):
    print(event)

/tmp/ipykernel_8405/3287479611.py:4: FutureWarning: The vertexai.Client class is deprecated. Please use agentplatform.Client instead.
  client = vertexai.Client(
INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '2.6.0', 'pydantic': '2.13.4', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.13.4'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.13.4']
INFO:vertexai_genai.agentengines:Using bucket qwiklabs-gcp-02-082f78356513-agent-staging-08cb277c
INFO:vertexai_genai.agentengines:Wrote to gs://qwiklabs-gcp-02-082f78356513-agent-staging-08cb277c/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://qwiklabs-gcp-02-082f78356513-agent-staging-08cb277c/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory ta

{'error_code': 'TypeError', 'error_message': "Expected agent to have tools and canonical_model attributes, but got <class 'google.adk.agents.llm_agent.LlmAgent'>", 'invocation_id': 'e-d7ab94a8-f983-4bba-84ed-2ff173e8fc49', 'author': 'Google', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'node_info': {'path': 'Google@1'}, 'id': '23c6c6df-b1c6-42eb-a176-945afc595252', 'timestamp': 1793114024.4028988}
{'code': 498, 'message': "Expected agent to have tools and canonical_model attributes, but got <class 'google.adk.agents.llm_agent.LlmAgent'>", 'errorCode': '498', 'errorMessage': "Expected agent to have tools and canonical_model attributes, but got <class 'google.adk.agents.llm_agent.LlmAgent'>"}


In [6]:
# @title Run this cell to delete the Agent Engine App
remote_agent.delete(force=True)

INFO:vertexai_genai.agentengines:Deleting AgentEngine resource: projects/899475961632/locations/us-central1/reasoningEngines/1907435245766967296
INFO:vertexai_genai.agentengines:Started AgentEngine delete operation: projects/899475961632/locations/us-central1/operations/5822396973721845760
